# 05 — Evaluation

## Predictive Maintenance for Aircraft Engines

### CRISP-DM Stage 5: Evaluation

**Dataset:** NASA C-MAPSS FD001
**Selected Model:** XGBoost
**Baseline Model:** Linear Regression

### Objective

Evaluate the selected predictive maintenance model on the official NASA FD001 test dataset.

This stage measures RUL prediction accuracy, examines late predictions near engine failure, and evaluates whether the model can support maintenance inspection decisions.

The evaluation follows the Business Understanding success criteria and uses the models and preprocessing methods established in the previous CRISP-DM stages.

### Evaluation Tasks

1. Predict RUL for the 100 test engines.
2. Compare XGBoost and Linear Regression using MAE and RMSE.
3. Visualize actual versus predicted RUL.
4. Analyze prediction errors across different RUL ranges.
5. Examine late predictions near failure.
6. Rank engines according to predicted RUL.
7. Review anomaly detection results.
8. Compare findings with the Business Understanding criteria.


## 1. Evaluation Setup and Reproducibility

Before evaluating the models, we verify that the Evaluation environment is consistent with the previous CRISP-DM stages.

We use the trained model artifacts produced during Stage 04 — Modeling without retraining or modifying them.

The Python environment is checked against the package versions documented in the Modeling stage. We also verify that all required model files are available.

This ensures that the Evaluation stage builds on the existing project rather than introducing a different modeling approach.

The official NASA C-MAPSS FD001 test dataset is reserved for the final evaluation and is not used for model training or hyperparameter tuning.


In [7]:

import sys
from pathlib import Path
import numpy as np
import pandas as pd
import sklearn
import joblib
import xgboost

# Locate the repository root.
current = Path.cwd().resolve()

if current.name == "notebooks" and current.parent.name == "05_evaluation":
    REPO_DIR = current.parent.parent
elif (current / "04_modeling").is_dir():
    REPO_DIR = current
else:
    raise RuntimeError(f"Unexpected working directory: {current}")

MODELS_DIR = REPO_DIR / "04_modeling" / "models"
EVAL_DIR = REPO_DIR / "05_evaluation"
OUTPUTS_DIR = EVAL_DIR / "outputs"
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

required_files = [
    MODELS_DIR / "selected_model.joblib",
    MODELS_DIR / "baseline_model.joblib",
    MODELS_DIR / "anomaly_detector.joblib",
    MODELS_DIR / "model_card.json",
]

print("Repository:", REPO_DIR)
print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("Scikit-learn:", sklearn.__version__)
print("XGBoost:", xgboost.__version__)

for file in required_files:
    print(("FOUND" if file.is_file() else "MISSING"),
          "-", file.relative_to(REPO_DIR))

assert all(file.is_file() for file in required_files), "Missing Modeling artifacts"
print("Model artifact paths verified.")


Repository: /Users/bugrahanyentimur/Desktop/CMPE412 Proje/CMPE412_Pedictive_Maintenance
Python: 3.12.10
NumPy: 2.4.6
Pandas: 2.2.3
Scikit-learn: 1.6.1
XGBoost: 3.4.1
FOUND - 04_modeling/models/selected_model.joblib
FOUND - 04_modeling/models/baseline_model.joblib
FOUND - 04_modeling/models/anomaly_detector.joblib
FOUND - 04_modeling/models/model_card.json
Model artifact paths verified.



## 2. Loading the Trained Models

In Stage 04 — Modeling, several machine learning algorithms were compared using the training and validation datasets.

Based on the validation results, XGBoost was selected as the final RUL prediction model. Linear Regression was retained as the baseline for performance comparison.

An Isolation Forest detector was also developed to identify unusual sensor behavior and support early maintenance warnings.

In this stage, we load the three saved model bundles directly from the Modeling artifacts:

- `selected_model.joblib`: Selected XGBoost model.
- `baseline_model.joblib`: Linear Regression baseline.
- `anomaly_detector.joblib`: Isolation Forest detector.
- `model_card.json`: Model configuration, selected features, and validation results.

The saved models are not retrained or modified during Evaluation.

We also verify the stored feature configurations to ensure that test data is processed consistently with the Modeling stage.


In [10]:

import json

selected_model = joblib.load(MODELS_DIR / "selected_model.joblib")
baseline_model = joblib.load(MODELS_DIR / "baseline_model.joblib")
anomaly_detector = joblib.load(MODELS_DIR / "anomaly_detector.joblib")

with open(MODELS_DIR / "model_card.json", "r", encoding="utf-8") as f:
    model_card = json.load(f)

feature_columns = model_card["feature_columns"]

print("Selected model:", model_card["selected_model"])
print("Selected model type:", type(selected_model).__name__)
print("Baseline model type:", type(baseline_model).__name__)
print("Anomaly detector type:", type(anomaly_detector).__name__)
print("Number of selected features:", len(feature_columns))
print("Models loaded successfully.")


Selected model: XGBoost (tuned)
Selected model type: dict
Baseline model type: dict
Anomaly detector type: dict
Number of selected features: 43
Models loaded successfully.


In [11]:

# Inspect the saved model bundles without modifying them.

for name, bundle in [
    ("Selected XGBoost", selected_model),
    ("Linear Regression Baseline", baseline_model),
    ("Isolation Forest", anomaly_detector),
]:
    print(f"\n{name}")
    print("-" * 40)

    for key, value in bundle.items():
        print(f"{key}: {type(value).__name__}")



Selected XGBoost
----------------------------------------
estimator: XGBRegressor
model_name: str
feature_columns: list
history_features: dict
target: str
trained_on: str

Linear Regression Baseline
----------------------------------------
estimator: Pipeline
model_name: str
feature_columns: list
target: str
trained_on: str

Isolation Forest
----------------------------------------
pipeline: Pipeline
threshold: float
feature_columns: list
history_features: dict
healthy_reference: str
threshold_rule: str
sustained_run: int
score: str


In [12]:

# Verify the expected input features of the saved models.

selected_features = selected_model["feature_columns"]
baseline_features = baseline_model["feature_columns"]
anomaly_features = anomaly_detector["feature_columns"]

print("XGBoost features:", len(selected_features))
print("Linear Regression features:", len(baseline_features))
print("Isolation Forest features:", len(anomaly_features))

print("\nXGBoost matches model card:",
      selected_features == feature_columns)

print("\nXGBoost history configuration:")
print(selected_model["history_features"])

print("\nIsolation Forest history configuration:")
print(anomaly_detector["history_features"])

assert selected_features == feature_columns, (
    "XGBoost feature list does not match the model card."
)

print("\nFeature configurations verified.")


XGBoost features: 43
Linear Regression features: 17
Isolation Forest features: 14

XGBoost matches model card: True

XGBoost history configuration:
{'mean_window': 10, 'slope_window': 20, 'sensors': ['sensor_2', 'sensor_3', 'sensor_4', 'sensor_7', 'sensor_8', 'sensor_9', 'sensor_11', 'sensor_12', 'sensor_13', 'sensor_14', 'sensor_15', 'sensor_17', 'sensor_20', 'sensor_21'], 'rule': 'per engine, ordered by cycle, current and earlier cycles only'}

Isolation Forest history configuration:
{'mean_window': 10, 'slope_window': 20, 'sensors': ['sensor_2', 'sensor_3', 'sensor_4', 'sensor_7', 'sensor_8', 'sensor_9', 'sensor_11', 'sensor_12', 'sensor_13', 'sensor_14', 'sensor_15', 'sensor_17', 'sensor_20', 'sensor_21'], 'rule': 'per engine, ordered by cycle, current and earlier cycles only'}

Feature configurations verified.



## 3. Official Test Data Verification

The Evaluation stage uses the official NASA C-MAPSS FD001 test dataset, which was reserved during the previous CRISP-DM stages.

To maintain consistency with Stage 02 — Data Understanding, we use the same NASA data source documented in `file_inventory.csv`.

### Data Source and Integrity

The official test dataset consists of two files:

- `test_FD001.txt`: Contains the operating cycles, settings, and sensor measurements of the test engines.
- `RUL_FD001.txt`: Contains the true Remaining Useful Life (RUL) at the last observed cycle of each test engine.

The original files were obtained from the NASA Turbofan Engine Degradation Simulation Data Set archive.

We verified both files using SHA-256 hashes recorded during Data Understanding. This confirms that the downloaded files match the dataset documented in the earlier stage.

### Dataset Verification Results

- Test observations: 13,096
- Test engines: 100
- Dataset columns: 26
- True RUL labels: 100
- Missing values: 0

### Evaluation Integrity

The test dataset was not used during model training, hyperparameter tuning, or model selection.

The saved models from Stage 04 — Modeling will be evaluated on the official test engines without retraining.

The true RUL labels will only be used to measure prediction accuracy and analyze maintenance-related errors.


In [15]:

# Inspect the dataset inventory from Data Understanding.

inventory_path = (
    REPO_DIR
    / "02_data_understanding"
    / "artifacts"
    / "data_understanding"
    / "file_inventory.csv"
)

inventory = pd.read_csv(inventory_path)

print("Dataset inventory:")
print(inventory.to_string(index=False))


Dataset inventory:
           file   bytes                                                           sha256                                                                                                                                                                              source
train_FD001.txt 3515356 963b5e22825b34d8b21c69e1aeb4af3e647050eb672ee8834ba4b5d91d2de0f8 https://phm-datasets.s3.amazonaws.com/NASA/6.+Turbofan+Engine+Degradation+Simulation+Data+Set.zip!6. Turbofan Engine Degradation Simulation Data Set/CMAPSSData.zip!train_FD001.txt
 test_FD001.txt 2228855 3cda7109ce17bafb5443f2ac926cfcf88154b941b8c4cf95eb55d1ddd6f52851  https://phm-datasets.s3.amazonaws.com/NASA/6.+Turbofan+Engine+Degradation+Simulation+Data+Set.zip!6. Turbofan Engine Degradation Simulation Data Set/CMAPSSData.zip!test_FD001.txt
  RUL_FD001.txt     429 a19c8ec94931949d0485bdc35118206e9c81c4547b422efb9cf86f4ceddbceca   https://phm-datasets.s3.amazonaws.com/NASA/6.+Turbofan+Engine+Degradation+Simulatio

In [22]:

import hashlib
import io
import ssl
import urllib.request
import zipfile
import certifi

# Use the official NASA source recorded in Data Understanding.
source_url = inventory.loc[
    inventory["file"] == "test_FD001.txt", "source"
].iloc[0].split("!")[0]

data_dir = EVAL_DIR / "data"
data_dir.mkdir(parents=True, exist_ok=True)

filenames = ["test_FD001.txt", "RUL_FD001.txt"]

# Check files against the original SHA-256 inventory.
def verify_file(filename, content):
    record = inventory.loc[inventory["file"] == filename].iloc[0]

    expected_hash = str(record["sha256"]).strip()
    actual_hash = hashlib.sha256(content).hexdigest()

    assert actual_hash == expected_hash, (
        f"SHA-256 mismatch for {filename}"
    )

    assert len(content) == int(record["bytes"]), (
        f"File size mismatch for {filename}"
    )

# Identify files that are missing or invalid.
files_to_download = []

for filename in filenames:
    path = data_dir / filename

    if path.is_file():
        content = path.read_bytes()

        try:
            verify_file(filename, content)
            print(f"VERIFIED (local): {filename}")
        except AssertionError:
            print(f"INVALID local file: {filename}")
            files_to_download.append(filename)
    else:
        print(f"MISSING: {filename}")
        files_to_download.append(filename)

# Download only if at least one file is missing or invalid.
if files_to_download:
    print("\nDownloading missing or invalid files from NASA...")

    ssl_context = ssl.create_default_context(
        cafile=certifi.where()
    )

    with urllib.request.urlopen(
        source_url,
        context=ssl_context,
        timeout=120
    ) as response:
        archive_bytes = response.read()

    with zipfile.ZipFile(io.BytesIO(archive_bytes)) as outer_zip:
        inner_name = next(
            name for name in outer_zip.namelist()
            if name.endswith("CMAPSSData.zip")
        )
        inner_bytes = outer_zip.read(inner_name)

    with zipfile.ZipFile(io.BytesIO(inner_bytes)) as inner_zip:
        for filename in files_to_download:
            content = inner_zip.read(filename)

            # Verify before saving.
            verify_file(filename, content)

            (data_dir / filename).write_bytes(content)
            print(f"VERIFIED (downloaded): {filename}")
else:
    print("\nAll official test files are already available.")

print("Official NASA FD001 test files are ready.")


VERIFIED (local): test_FD001.txt
VERIFIED (local): RUL_FD001.txt

All official test files are already available.
Official NASA FD001 test files are ready.


In [19]:

# Load the official NASA FD001 test data.

data_dir = EVAL_DIR / "data"

columns = (
    ["unit", "cycle"]
    + [f"setting_{i}" for i in range(1, 4)]
    + [f"sensor_{i}" for i in range(1, 22)]
)

test_df = pd.read_csv(
    data_dir / "test_FD001.txt",
    sep=r"\s+",
    header=None,
    names=columns
)

rul_df = pd.read_csv(
    data_dir / "RUL_FD001.txt",
    sep=r"\s+",
    header=None,
    names=["true_rul"]
)

print("Test data shape:", test_df.shape)
print("Number of test engines:", test_df["unit"].nunique())
print("Number of RUL labels:", len(rul_df))
print("Missing values:", test_df.isna().sum().sum())

assert test_df["unit"].nunique() == 100
assert len(rul_df) == 100
assert test_df.shape[1] == 26

print("\nOfficial FD001 test data loaded successfully.")


Test data shape: (13096, 26)
Number of test engines: 100
Number of RUL labels: 100
Missing values: 0

Official FD001 test data loaded successfully.



## 4. Test Data Preparation and History Features

### Objective

Prepare the official NASA C-MAPSS FD001 test data for the trained models without changing the preprocessing and feature engineering methods established in the previous CRISP-DM stages.

### Consistency with Previous Stages

Stage 03 — Data Preparation established the dataset structure, feature selection, and engine-level separation.

Stage 04 — Modeling introduced historical sensor features to improve RUL prediction. We reproduce the same feature engineering procedure on the official test trajectories.

### History Feature Engineering

The selected XGBoost model uses 43 features:

- 14 current sensor measurements.
- 14 trailing means calculated over the last 10 cycles.
- 14 least-squares slopes calculated over the last 20 cycles.
- 1 operating cycle feature.

Historical features are calculated separately for each engine using only the current and previous observations.

At the beginning of each engine trajectory, shorter available windows are used. The slope is set to zero when there is insufficient variation in cycle values.

This procedure prevents future information and measurements from other engines from entering the feature calculations.

### Model Input Consistency

Each saved model uses its own feature configuration:

- XGBoost: 43 features.
- Linear Regression baseline: 17 features.
- Isolation Forest: 14 features.

For final RUL prediction, we will use the last observed cycle of each of the 100 test engines.

No model retraining, hyperparameter tuning, or feature selection will be performed using the official test set.


In [20]:

# Reproduce the history features from Stage 04 Modeling.

SENSORS = selected_model["history_features"]["sensors"]
MEAN_WINDOW = selected_model["history_features"]["mean_window"]
SLOPE_WINDOW = selected_model["history_features"]["slope_window"]

def add_history_features(df):
    out = df.copy()

    def trailing_mean(series, window):
        return series.groupby(out["unit"]).transform(
            lambda v: v.rolling(window, min_periods=1).mean()
        )

    t = out["cycle"].astype(float)
    m_t = trailing_mean(t, SLOPE_WINDOW)
    var_t = trailing_mean(t * t, SLOPE_WINDOW) - m_t ** 2

    new_columns = {}

    for s in SENSORS:
        x = out[s]

        new_columns[f"mean{MEAN_WINDOW}_{s}"] = trailing_mean(
            x, MEAN_WINDOW
        )

        cov_tx = (
            trailing_mean(t * x, SLOPE_WINDOW)
            - m_t * trailing_mean(x, SLOPE_WINDOW)
        )

        new_columns[f"slope{SLOPE_WINDOW}_{s}"] = np.where(
            var_t > 1e-9,
            cov_tx / var_t,
            0.0
        )

    return pd.concat(
        [out, pd.DataFrame(new_columns, index=out.index)],
        axis=1
    )

# Sort observations by engine and cycle before feature creation.
test_sorted = test_df.sort_values(
    ["unit", "cycle"]
).reset_index(drop=True)

test_history = add_history_features(test_sorted)

# Verify that all selected XGBoost features exist.
missing_features = [
    col for col in selected_features
    if col not in test_history.columns
]

assert not missing_features, (
    f"Missing features: {missing_features}"
)

assert test_history[selected_features].notna().all().all()

print("Test observations:", len(test_history))
print("Test engines:", test_history["unit"].nunique())
print("XGBoost features verified:", len(selected_features))
print("History features created successfully.")


Test observations: 13096
Test engines: 100
XGBoost features verified: 43
History features created successfully.



## 5. Final Test Predictions

### Objective

Generate Remaining Useful Life (RUL) predictions for the 100 official NASA C-MAPSS FD001 test engines using the trained models from Stage 04 — Modeling.

### Prediction Procedure

For each test engine, we select its last observed operating cycle after applying the same feature engineering procedure used during Modeling.

We use two previously trained models:

- **XGBoost:** The selected model, using 43 input features.
- **Linear Regression:** The baseline model, using 17 input features.

Each model receives its own saved feature columns in the original order.

The models are used without retraining or hyperparameter tuning.

### Evaluation Integrity

The official RUL labels are kept separate from the prediction inputs and are used only to calculate evaluation metrics.

We will compare the predictions with the true RUL values to determine whether the selected XGBoost model provides reliable information for maintenance inspection planning.

All predictions will be recorded for further error analysis and business evaluation.


In [23]:

# Select the last observed cycle of each test engine.
test_last = (
    test_history
    .sort_values(["unit", "cycle"])
    .groupby("unit", sort=True)
    .tail(1)
    .sort_values("unit")
    .reset_index(drop=True)
)

# Verify the test engine order.
assert len(test_last) == 100
assert test_last["unit"].tolist() == list(range(1, 101))

# Use each model's original feature configuration.
X_xgb = test_last[selected_features]
X_baseline = test_last[baseline_features]

assert X_xgb.shape == (100, 43)
assert X_baseline.shape == (100, 17)

# Generate predictions using the saved trained estimators.
xgb_predictions = selected_model["estimator"].predict(X_xgb)
baseline_predictions = baseline_model["estimator"].predict(
    X_baseline
)

# The official NASA RUL labels follow engine order.
prediction_results = pd.DataFrame({
    "unit": test_last["unit"].to_numpy(),
    "last_cycle": test_last["cycle"].to_numpy(),
    "true_rul": rul_df["true_rul"].to_numpy(),
    "xgb_predicted_rul": xgb_predictions,
    "baseline_predicted_rul": baseline_predictions,
})

# Verify prediction completeness.
assert prediction_results.shape[0] == 100
assert np.isfinite(
    prediction_results[
        ["xgb_predicted_rul", "baseline_predicted_rul"]
    ].to_numpy()
).all()

print("Test engines evaluated:", len(prediction_results))
print("XGBoost input shape:", X_xgb.shape)
print("Baseline input shape:", X_baseline.shape)

print("\nFirst five predictions:")
print(prediction_results.head().round(2).to_string(index=False))

print("\nFinal test predictions generated successfully.")


Test engines evaluated: 100
XGBoost input shape: (100, 43)
Baseline input shape: (100, 17)

First five predictions:
 unit  last_cycle  true_rul  xgb_predicted_rul  baseline_predicted_rul
    1          31       112         121.910004                  159.28
    2          49        98         119.940002                  122.22
    3         126        69          43.090000                   85.44
    4         106        82          78.339996                   99.38
    5          98        91          92.779999                  110.92

Final test predictions generated successfully.



## 6. Model Performance Evaluation

### Objective

Evaluate the predictive accuracy of the selected XGBoost model and compare its performance with the Linear Regression baseline on the official NASA C-MAPSS FD001 test dataset.

### Evaluation Metrics

We use two regression metrics:

- **Mean Absolute Error (MAE):** Measures the average absolute difference between predicted and actual RUL values.
- **Root Mean Squared Error (RMSE):** Measures prediction error while assigning greater weight to large errors.

Lower values indicate better predictive accuracy.

### Evaluation Strategy

Both models are evaluated on the same 100 test engines using their previously saved parameters.

In addition to overall MAE and RMSE, we will examine errors across different RUL ranges, with particular attention to engines approaching failure.

The results will be compared with the success criteria established during Stage 01 — Business Understanding.

No model retraining or hyperparameter tuning is performed using the official test dataset.


In [24]:

from sklearn.metrics import mean_absolute_error, mean_squared_error

# Use the official test labels only for evaluation.
y_true = prediction_results["true_rul"].to_numpy()

model_predictions = {
    "Linear Regression (baseline)": prediction_results[
        "baseline_predicted_rul"
    ].to_numpy(),
    "XGBoost (selected)": prediction_results[
        "xgb_predicted_rul"
    ].to_numpy(),
}

metrics_results = []

for model_name, y_pred in model_predictions.items():
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))

    metrics_results.append({
        "Model": model_name,
        "MAE": mae,
        "RMSE": rmse
    })

metrics_df = pd.DataFrame(metrics_results)

baseline_mae = metrics_df.loc[
    metrics_df["Model"] == "Linear Regression (baseline)", "MAE"
].iloc[0]

xgb_mae = metrics_df.loc[
    metrics_df["Model"] == "XGBoost (selected)", "MAE"
].iloc[0]

mae_improvement = (
    (baseline_mae - xgb_mae) / baseline_mae
) * 100

print("Official NASA FD001 Test Performance")
print("-" * 45)
print(metrics_df.round(3).to_string(index=False))

print(f"\nXGBoost MAE improvement over baseline: {mae_improvement:.2f}%")

print("\nTest performance evaluation completed.")


Official NASA FD001 Test Performance
---------------------------------------------
                       Model    MAE   RMSE
Linear Regression (baseline) 25.603 31.969
          XGBoost (selected) 10.974 15.258

XGBoost MAE improvement over baseline: 57.14%

Test performance evaluation completed.



### Test Performance Results and Interpretation

The selected XGBoost model was evaluated on the official NASA C-MAPSS FD001 test set containing 100 engines.

The following results were obtained:

| Model | MAE | RMSE |
|---|---:|---:|
| Linear Regression (Baseline) | 25.603 | 31.969 |
| XGBoost (Selected) | 10.974 | 15.258 |

XGBoost reduced the overall MAE by 57.14% compared with the Linear Regression baseline.

These findings support the model selection decision made during Stage 04 — Modeling.

However, overall prediction accuracy alone is not sufficient to determine whether the model is suitable for maintenance planning.

In predictive maintenance, overestimating the remaining useful life of an engine close to failure can lead to delayed inspections.

Therefore, the next analysis examines prediction errors across different RUL ranges and identifies potentially costly late predictions.



### Prediction Error Analysis by RUL Range

Overall MAE and RMSE provide useful measures of prediction accuracy, but they do not fully describe maintenance-related risks.

In predictive maintenance, prediction errors have different consequences depending on how close an engine is to failure.

To examine this, we divide the official test engines into four groups based on their true Remaining Useful Life (RUL):

- **RUL ≤ 30:** Critical engines approaching failure.
- **31–90 cycles:** Engines within an intermediate maintenance planning horizon.
- **91–125 cycles:** Engines approaching the upper limit of the Modeling decision horizon.
- **RUL > 125:** Engines outside the capped target range used during Modeling.

For each group, we calculate MAE and RMSE for both XGBoost and Linear Regression.

We also measure the frequency of late predictions, where predicted RUL exceeds actual RUL.

This analysis helps determine whether the selected model supports timely inspections and identifies potential limitations in maintenance decision-making.


In [25]:

# Evaluate prediction errors across true RUL ranges.

rul_bins = [-np.inf, 30, 90, 125, np.inf]
rul_labels = ["RUL <= 30", "31-90", "91-125", "RUL > 125"]

evaluation_df = prediction_results.copy()

evaluation_df["rul_band"] = pd.cut(
    evaluation_df["true_rul"],
    bins=rul_bins,
    labels=rul_labels
)

band_results = []

for band in rul_labels:
    subset = evaluation_df[
        evaluation_df["rul_band"] == band
    ]

    for model_name, pred_col in [
        ("Linear Regression", "baseline_predicted_rul"),
        ("XGBoost", "xgb_predicted_rul")
    ]:
        if subset.empty:
            continue

        y_true_band = subset["true_rul"]
        y_pred_band = subset[pred_col]

        mae = mean_absolute_error(y_true_band, y_pred_band)
        rmse = np.sqrt(
            mean_squared_error(y_true_band, y_pred_band)
        )

        late_rate = (
            (y_pred_band > y_true_band).mean() * 100
        )

        band_results.append({
            "RUL Range": band,
            "Model": model_name,
            "Engines": len(subset),
            "MAE": round(mae, 3),
            "RMSE": round(rmse, 3),
            "Late Predictions (%)": round(late_rate, 2)
        })

band_metrics_df = pd.DataFrame(band_results)

print("Prediction Error Analysis by RUL Range")
print("-" * 60)
print(band_metrics_df.to_string(index=False))

print("\nRUL band distribution:")
print(
    evaluation_df["rul_band"]
    .value_counts()
    .reindex(rul_labels)
    .to_string()
)

print("\nRUL range analysis completed.")


Prediction Error Analysis by RUL Range
------------------------------------------------------------
RUL Range             Model  Engines    MAE   RMSE  Late Predictions (%)
RUL <= 30 Linear Regression       25 21.255 28.195                 80.00
RUL <= 30           XGBoost       25  4.598  7.533                 72.00
    31-90 Linear Regression       29 34.272 39.242                 96.55
    31-90           XGBoost       29 11.598 15.806                 62.07
   91-125 Linear Regression       35 23.391 29.916                 91.43
   91-125           XGBoost       35 11.471 15.708                 57.14
RUL > 125 Linear Regression       11 19.664 24.023                 54.55
RUL > 125           XGBoost       11 22.234 23.318                  0.00

RUL band distribution:
rul_band
RUL <= 30    25
31-90        29
91-125       35
RUL > 125    11

RUL range analysis completed.



### RUL Range Results and Interpretation

The official FD001 test set contains 100 engines distributed across four RUL ranges.

| True RUL Range | Engines | XGBoost MAE | XGBoost RMSE | Late Predictions |
|---|---:|---:|---:|---:|
| RUL <= 30 | 25 | 4.598 | 7.533 | 72.00% |
| 31–90 | 29 | 11.598 | 15.806 | 62.07% |
| 91–125 | 35 | 11.471 | 15.708 | 57.14% |
| RUL > 125 | 11 | 22.234 | 23.318 | 0.00% |

### Key Findings

**Critical engines (RUL <= 30):**

XGBoost achieved an MAE of 4.598 cycles, compared with 21.255 cycles for the Linear Regression baseline.

This indicates substantially better prediction accuracy for engines approaching failure.

However, 72% of critical engines received predictions higher than their actual remaining useful life.

**Intermediate RUL ranges:**

For engines with RUL between 31 and 125 cycles, XGBoost achieved MAE values of approximately 11–12 cycles.

Late predictions occurred in more than half of these engines.

**Long-life engines (RUL > 125):**

XGBoost achieved an MAE of 22.234 cycles, reflecting reduced accuracy outside the capped training target range.

This behavior is consistent with the 125-cycle target cap established during Modeling.

### Maintenance Implications

Although XGBoost outperformed the baseline in the most critical RUL range, the high frequency of late predictions requires further investigation.

A late prediction may delay an inspection, especially when an engine is close to failure.

Therefore, the next analysis examines the magnitude of late prediction errors rather than considering their frequency alone.

The model's suitability for maintenance planning will be assessed against the Business Understanding success criteria.
